# STIR-Net first single-sample overfit

This notebook starts entirely from the artifacts saved by the preparation notebook:

- raw microscopy movie;
- imperfect CC-only instance masks;
- GT instance labels;
- marker masks;
- cached target-frame raw normalization, foreground, per-instance EDT, and boundary;
- Trackastra pass-1 graph.

It keeps the full cell population together. Only empty acquisition margins are removed.

The notebook includes temporary memory-safety patches for the first overfit experiment:
AMP-safe segmented softmax, lower-peak axis-factorized convolution, chunked co-reasoning
attention, and a reduced-width debug profile. These are deliberately isolated here so the
experiment can establish whether the same sample can be learned before production memory
refactoring is committed to the model source.

**Before running this notebook, finish or shut down the preparation notebook kernel so it does not keep GPU memory allocated.**


In [ ]:
from pathlib import Path
import gc
import json
import math
import pickle
import time

import numpy as np
import torch


def find_repo_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()

    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "learned").exists():
            return path

    raise RuntimeError("Could not locate repository root.")


PROJECT_ROOT = find_repo_root()

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

TRACKASTRA_DIR = DATA_DIR / "trackastra"
STIRNET_SOURCE_DIR = DATA_DIR / "stirnet_source"

print("Project root:", PROJECT_ROOT)
print("Data dir    :", DATA_DIR)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    print(
        "VRAM        :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )


## Load all saved inputs


In [ ]:
frame_numbers = np.load(
    DATA_DIR / "frame_numbers.npy",
)

raw_movie = np.load(
    DATA_DIR / "raw_movie.npy",
    mmap_mode="r",
)

processed_movie = np.load(
    DATA_DIR / "processed_movie.npy",
    mmap_mode="r",
)

binary_movie = np.load(
    DATA_DIR / "binary_movie.npy",
    mmap_mode="r",
)

instance_movie = np.load(
    DATA_DIR / "instance_movie.npy",
    mmap_mode="r",
)

markers_movie = np.load(
    DATA_DIR / "markers_movie.npy",
    mmap_mode="r",
)

gt_movie = np.load(
    DATA_DIR / "gt_movie.npy",
    mmap_mode="r",
)

with open(
    DATA_DIR / "metadata.json",
    "r",
    encoding="utf-8",
) as f:
    metadata = json.load(f)

SPACING_ZYX_UM = tuple(
    float(x)
    for x in metadata["spacing_zyx_um"]
)

with open(
    TRACKASTRA_DIR / "track_graph.pkl",
    "rb",
) as f:
    track_graph = pickle.load(f)

# Tracked masks are kept memory-mapped for inspection/debugging.
masks_tracked = np.load(
    TRACKASTRA_DIR / "masks_tracked.npy",
    mmap_mode="r",
)

raw_norm_target_full = np.load(
    STIRNET_SOURCE_DIR / "raw_norm_target.npy",
    mmap_mode="r",
)

foreground_target_full = np.load(
    STIRNET_SOURCE_DIR / "foreground_target.npy",
    mmap_mode="r",
)

edt_target_full = np.load(
    STIRNET_SOURCE_DIR / "edt_target.npy",
    mmap_mode="r",
)

boundary_target_full = np.load(
    STIRNET_SOURCE_DIR / "boundary_target.npy",
    mmap_mode="r",
)

marker_heatmap_target_full = np.load(
    STIRNET_SOURCE_DIR / "marker_heatmap_target.npy",
    mmap_mode="r",
)

dref_um = float(
    np.load(
        STIRNET_SOURCE_DIR / "dref_um.npy",
    )
)

TARGET_LOCAL_T = 2
TARGET_FRAME = int(frame_numbers[TARGET_LOCAL_T])

print("Frames       :", frame_numbers.tolist())
print("Raw movie    :", raw_movie.shape, raw_movie.dtype)
print("Processed    :", processed_movie.shape, processed_movie.dtype)
print("Binary mask  :", binary_movie.shape, binary_movie.dtype)
print("Instances    :", instance_movie.shape, instance_movie.dtype)
print("GT           :", gt_movie.shape, gt_movie.dtype)
print("Track nodes  :", track_graph.number_of_nodes())
print("Track edges  :", track_graph.number_of_edges())
print("Target frame :", TARGET_FRAME)
print("d_ref        :", f"{dref_um:.4f} µm")
print("Spacing ZYX  :", SPACING_ZYX_UM)


## Runtime patches for the first overfit

These patches are experiment-local. They should later be converted into proper source-code
changes with tests once the overfit test succeeds.


In [ ]:
# AMP-safe segmented softmax.

import learned.stirnet.model.graph_encoder as graph_encoder
import learned.stirnet.model.temporal_hypotheses as temporal_hypotheses


def safe_segment_softmax(
    scores: torch.Tensor,
    index: torch.Tensor,
    num_segments: int,
) -> torch.Tensor:
    if scores.numel() == 0:
        return scores

    original_dtype = scores.dtype

    if scores.dtype in (
        torch.float16,
        torch.bfloat16,
    ):
        work = scores.float()
    else:
        work = scores

    h = work.shape[1]
    idx = index[:, None].expand(-1, h)

    max_buf = torch.full(
        (num_segments, h),
        -torch.inf,
        device=work.device,
        dtype=work.dtype,
    )

    max_buf.scatter_reduce_(
        0,
        idx,
        work,
        reduce="amax",
        include_self=True,
    )

    stable = work - max_buf[index]
    ex = torch.exp(stable)

    denom = torch.zeros(
        (num_segments, h),
        device=work.device,
        dtype=work.dtype,
    )

    denom.index_add_(
        0,
        index,
        ex,
    )

    result = (
        ex
        / denom[index].clamp_min(1e-8)
    )

    return result.to(original_dtype)


graph_encoder.segment_softmax = safe_segment_softmax
temporal_hypotheses.segment_softmax = safe_segment_softmax

print("Installed AMP-safe segment_softmax.")


In [ ]:
# Lower-peak-memory axis-factorized convolution.
# The computation remains z + y + x followed by the 1x1 fuse.

import torch.nn.functional as F

from learned.stirnet.model.blocks import AxisFactorizedConv


def low_memory_axis_forward(
    self,
    x,
    acquisition_embedding,
):
    gates = torch.sigmoid(
        self.gate(acquisition_embedding)
    ).view(
        x.shape[0],
        3,
        self.channels,
        1,
        1,
        1,
    )

    out = (
        self.conv_z(x) * gates[:, 0]
        + self.conv_y(x) * gates[:, 1]
    )

    out = (
        out
        + self.conv_x(x) * gates[:, 2]
    )

    return self.fuse(out)


AxisFactorizedConv.forward = low_memory_axis_forward

print("Installed lower-peak AxisFactorizedConv.")


In [ ]:
# Chunked all-cell co-reasoning attention.
#
# The current source computes all temporal-spatial positional pairs before
# radius masking. These replacements preserve the same global sample and
# attention equations, but bound transient memory.

from learned.stirnet.model.attention import LocalPhysicalCrossAttention


TEMPORAL_QUERY_CHUNK = 1
SPATIAL_TOKEN_CHUNK = 2048


def chunked_temporal_reads_spatial(
    self,
    temporal,
    temporal_ref_um,
    salience,
    spatial,
    spatial_pos_um,
    temporal_batch,
    dref_um,
    spatial_padding_mask=None,
    base_radius_dref=1.5,
):
    B = spatial.shape[0]
    out = torch.zeros_like(temporal)

    for b in range(B):
        ids = torch.nonzero(
            temporal_batch == b,
            as_tuple=False,
        ).flatten()

        if ids.numel() == 0:
            continue

        t = temporal[ids]
        tref = temporal_ref_um[ids]
        sal = salience[ids, 0]

        s = spatial[b]
        spos = spatial_pos_um[b]

        k = self._split(
            self.s_k(s)
        ).permute(1, 0, 2)

        v = self._split(
            self.s_v(s)
        ).permute(1, 0, 2)

        padding_flat = None

        if spatial_padding_mask is not None:
            padding_flat = spatial_padding_mask[b].reshape(-1)

        messages = []

        for start in range(
            0,
            len(ids),
            TEMPORAL_QUERY_CHUNK,
        ):
            end = min(
                start + TEMPORAL_QUERY_CHUNK,
                len(ids),
            )

            t_chunk = t[start:end]
            tref_chunk = tref[start:end]
            sal_chunk = sal[start:end]

            q = self._split(
                self.t_q(t_chunk)
            ).permute(1, 0, 2)

            logits = torch.einsum(
                "hmd,hnd->hmn",
                q,
                k,
            ) / math.sqrt(self.head_dim)

            delta = (
                spos[None, :, :]
                - tref_chunk[:, None, :]
            )

            bias = self.pos_bias(
                delta,
                dref_um[b],
            ).permute(2, 0, 1)

            logits = logits + bias

            dist = torch.linalg.vector_norm(
                delta,
                dim=-1,
            )

            radius = (
                base_radius_dref
                + sal_chunk
            ).clamp_max(
                base_radius_dref + 1.0
            ) * dref_um[b]

            invalid = (
                dist
                > radius[:, None]
            )

            if padding_flat is not None:
                invalid = (
                    invalid
                    | padding_flat[None, :]
                )

            logits = logits.masked_fill(
                invalid[None],
                -1e4,
            )

            weights = torch.softmax(
                logits,
                dim=-1,
            )

            weights = F.dropout(
                weights,
                self.dropout,
                self.training,
            )

            msg = torch.einsum(
                "hmn,hnd->hmd",
                weights,
                v,
            )

            msg = (
                msg
                .permute(1, 0, 2)
                .reshape(
                    end - start,
                    self.d_model,
                )
            )

            messages.append(
                self.t_out(msg).to(
                    dtype=out.dtype
                )
            )

        out = out.index_copy(
            0,
            ids,
            torch.cat(messages, dim=0),
        )

    return out


def chunked_spatial_reads_temporal(
    self,
    spatial,
    spatial_pos_um,
    temporal,
    temporal_ref_um,
    salience,
    reliability,
    temporal_batch,
    dref_um,
    spatial_padding_mask=None,
    base_radius_dref=1.5,
):
    B = spatial.shape[0]

    sal_scale = F.softplus(
        self.salience_scale
    )[:, None, None]

    rel_scale = (
        self.reliability_scale[
            :, None, None
        ]
    )

    batch_outputs = []

    for b in range(B):
        ids = torch.nonzero(
            temporal_batch == b,
            as_tuple=False,
        ).flatten()

        s = spatial[b]
        spos = spatial_pos_um[b]

        if ids.numel() == 0:
            batch_outputs.append(
                torch.zeros_like(s)
            )
            continue

        t = temporal[ids]
        tref = temporal_ref_um[ids]
        sal = salience[ids, 0]

        rel = reliability[
            ids,
            0,
        ].clamp_min(1e-4)

        k = self._split(
            self.t_k(t)
        ).permute(1, 0, 2)

        v = self._split(
            self.t_v(t)
        ).permute(1, 0, 2)

        padding_flat = None

        if spatial_padding_mask is not None:
            padding_flat = spatial_padding_mask[b].reshape(-1)

        spatial_messages = []

        for start in range(
            0,
            s.shape[0],
            SPATIAL_TOKEN_CHUNK,
        ):
            end = min(
                start + SPATIAL_TOKEN_CHUNK,
                s.shape[0],
            )

            s_chunk = s[start:end]
            spos_chunk = spos[start:end]

            q = self._split(
                self.s_q(s_chunk)
            ).permute(1, 0, 2)

            logits = torch.einsum(
                "hnd,hmd->hnm",
                q,
                k,
            ) / math.sqrt(self.head_dim)

            delta = (
                tref[None, :, :]
                - spos_chunk[:, None, :]
            )

            bias = self.pos_bias(
                delta,
                dref_um[b],
            ).permute(2, 0, 1)

            logits = (
                logits
                + bias
                + sal_scale
                * sal[None, None, :]
                + rel_scale
                * torch.log(rel)[
                    None, None, :
                ]
            )

            dist = torch.linalg.vector_norm(
                delta,
                dim=-1,
            )

            radius = (
                base_radius_dref
                + sal
            ).clamp_max(
                base_radius_dref + 1.0
            ) * dref_um[b]

            invalid = (
                dist
                > radius[None, :]
            )

            logits = logits.masked_fill(
                invalid[None],
                -1e4,
            )

            valid_any = (
                ~invalid
            ).any(dim=-1)

            weights = torch.softmax(
                logits,
                dim=-1,
            )

            weights = (
                weights
                * valid_any[
                    None,
                    :,
                    None,
                ]
            )

            weights = F.dropout(
                weights,
                self.dropout,
                self.training,
            )

            msg = torch.einsum(
                "hnm,hmd->hnd",
                weights,
                v,
            )

            msg = (
                msg
                .permute(1, 0, 2)
                .reshape(
                    end - start,
                    self.d_model,
                )
            )

            msg = self.s_out(msg).to(
                dtype=s.dtype
            )

            if padding_flat is not None:
                msg = msg.masked_fill(
                    padding_flat[
                        start:end,
                        None,
                    ],
                    0,
                )

            spatial_messages.append(msg)

        batch_outputs.append(
            torch.cat(
                spatial_messages,
                dim=0,
            )
        )

    return torch.stack(
        batch_outputs,
        dim=0,
    )


LocalPhysicalCrossAttention.temporal_reads_spatial = (
    chunked_temporal_reads_spatial
)

LocalPhysicalCrossAttention.spatial_reads_temporal = (
    chunked_spatial_reads_temporal
)

print("Installed chunked co-reasoning attention.")
print("Temporal query chunk:", TEMPORAL_QUERY_CHUNK)
print("Spatial token chunk :", SPATIAL_TOKEN_CHUNK)


## Keep all cells; remove only empty acquisition margins


In [ ]:
spacing = np.asarray(
    SPACING_ZYX_UM,
    dtype=np.float32,
)

full_shape = np.asarray(
    raw_movie.shape[-3:],
    dtype=int,
)

ROI_MARGIN_UM = 12.0

margin_vox = np.ceil(
    ROI_MARGIN_UM / spacing
).astype(int)

global_lo = full_shape.copy()
global_hi = np.zeros(3, dtype=int)

for t in range(len(frame_numbers)):
    current_fg = (
        np.asarray(instance_movie[t]) > 0
    )

    gt_fg = (
        np.asarray(gt_movie[t]) > 0
    )

    foreground = current_fg | gt_fg
    coords = np.where(foreground)

    if len(coords[0]) == 0:
        continue

    lo = np.asarray(
        [axis.min() for axis in coords],
        dtype=int,
    )

    hi = np.asarray(
        [axis.max() + 1 for axis in coords],
        dtype=int,
    )

    global_lo = np.minimum(
        global_lo,
        lo,
    )

    global_hi = np.maximum(
        global_hi,
        hi,
    )

    del current_fg, gt_fg, foreground, coords

roi_lo = np.maximum(
    global_lo - margin_vox,
    0,
)

roi_hi = np.minimum(
    global_hi + margin_vox,
    full_shape,
)

ROI_SLICES = tuple(
    slice(int(lo), int(hi))
    for lo, hi in zip(
        roi_lo,
        roi_hi,
    )
)

ROI_SHAPE = tuple(
    int(hi - lo)
    for lo, hi in zip(
        roi_lo,
        roi_hi,
    )
)

print("Full shape:", tuple(full_shape))
print("ROI shape :", ROI_SHAPE)
print(
    "Voxels retained:",
    f"{100 * np.prod(ROI_SHAPE) / np.prod(full_shape):.1f}%",
)


In [ ]:
# Materialize only the target-frame all-cell ROI.
# EDT/boundary/raw normalization are loaded from the preparation cache.

raw_target = np.asarray(
    raw_movie[TARGET_LOCAL_T][ROI_SLICES]
).copy()

current_target = np.asarray(
    instance_movie[TARGET_LOCAL_T][ROI_SLICES]
).astype(np.int32, copy=True)

gt_target = np.asarray(
    gt_movie[TARGET_LOCAL_T][ROI_SLICES]
).astype(np.int32, copy=True)

raw_norm_target = np.asarray(
    raw_norm_target_full[ROI_SLICES]
).astype(np.float32, copy=True)

foreground_target = np.asarray(
    foreground_target_full[ROI_SLICES]
).astype(np.float32, copy=True)

edt_target = np.asarray(
    edt_target_full[ROI_SLICES]
).astype(np.float32, copy=True)

boundary_target = np.asarray(
    boundary_target_full[ROI_SLICES]
).astype(np.float32, copy=True)

marker_target = np.asarray(
    marker_heatmap_target_full[ROI_SLICES]
).astype(np.float32, copy=True)

spatial_inputs_np = np.stack(
    [
        raw_norm_target,
        foreground_target,
        edt_target,
        boundary_target,
        marker_target,
    ],
    axis=0,
)

current_count_full = int(
    np.count_nonzero(
        np.unique(
            instance_movie[TARGET_LOCAL_T]
        ) > 0
    )
)

gt_count_full = int(
    np.count_nonzero(
        np.unique(
            gt_movie[TARGET_LOCAL_T]
        ) > 0
    )
)

current_count_roi = int(
    np.count_nonzero(
        np.unique(current_target) > 0
    )
)

gt_count_roi = int(
    np.count_nonzero(
        np.unique(gt_target) > 0
    )
)

assert current_count_roi == current_count_full
assert gt_count_roi == gt_count_full

print("Spatial inputs:", spatial_inputs_np.shape)
print("Current cells :", current_count_roi)
print("GT cells      :", gt_count_roi)
print("EDT range     :", float(edt_target.min()), float(edt_target.max()))


## Convert saved Trackastra graph to STIR-Net temporal tensors


In [ ]:
from learned.stirnet.data.sample_builder import robust_normalize
from learned.stirnet.data.targets import extract_instance_metadata
from learned.stirnet.data.graph_builder import (
    DetectionRecord,
    AssociationRecord,
    build_temporal_graph,
)


roi_lo_float = roi_lo.astype(np.float32)
roi_shape_arr = np.asarray(
    ROI_SHAPE,
    dtype=np.float32,
)

roi_center_um = (
    0.5
    * (roi_shape_arr - 1.0)
    * spacing
)

full_shape_float = full_shape.astype(
    np.float32
)

node_position_abs_um = {}

for node_id, data in track_graph.nodes(data=True):
    coords_zyx = np.asarray(
        data["coords"],
        dtype=np.float32,
    )

    node_position_abs_um[int(node_id)] = (
        coords_zyx * spacing
    )


def mean_velocity(
    node_id,
    neighbours,
    forward=True,
):
    node_id = int(node_id)

    if not neighbours:
        return np.zeros(
            3,
            dtype=np.float32,
        )

    t0 = int(
        track_graph.nodes[node_id]["time"]
    )

    p0 = node_position_abs_um[node_id]
    velocities = []

    for other in neighbours:
        other = int(other)

        t1 = int(
            track_graph.nodes[other]["time"]
        )

        p1 = node_position_abs_um[other]
        dt = abs(t1 - t0)

        if dt == 0:
            continue

        if forward:
            velocity = (p1 - p0) / dt
        else:
            velocity = (p0 - p1) / dt

        velocities.append(velocity)

    if not velocities:
        return np.zeros(
            3,
            dtype=np.float32,
        )

    return np.mean(
        velocities,
        axis=0,
    ).astype(np.float32)


In [ ]:
detection_records = []

for local_t in range(len(frame_numbers)):
    labels = np.asarray(
        instance_movie[local_t][ROI_SLICES]
    ).astype(np.int32, copy=False)

    raw = np.asarray(
        raw_movie[local_t][ROI_SLICES]
    )

    markers = np.asarray(
        markers_movie[local_t][ROI_SLICES]
    )

    raw_norm = robust_normalize(raw)

    marker_heatmap_frame = (
        markers > 0
    ).astype(np.float32)

    instance_meta = extract_instance_metadata(
        labels,
        raw_norm,
        SPACING_ZYX_UM,
        dref_um,
        marker_heatmap_frame,
    )

    ids = instance_meta.ids.cpu().numpy()
    features = instance_meta.features.cpu().numpy()

    id_to_row = {
        int(instance_id): row
        for row, instance_id in enumerate(ids)
    }

    for node_id, node_data in track_graph.nodes(data=True):
        if int(node_data["time"]) != local_t:
            continue

        label_id = int(node_data["label"])

        if label_id not in id_to_row:
            continue

        row = id_to_row[label_id]

        coords_full = np.asarray(
            node_data["coords"],
            dtype=np.float32,
        )

        coords_roi = (
            coords_full
            - roi_lo_float
        )

        position_rel_um = (
            coords_roi * spacing
            - roi_center_um
        )

        component = (
            labels == label_id
        )

        component_coords = np.argwhere(
            component
        )

        physical_volume_um3 = (
            len(component_coords)
            * float(np.prod(spacing))
        )

        feat = features[row]

        bbox_um = (
            feat[1:4] * dref_um
        )

        pca_axes_um = (
            feat[4:7] * dref_um
        )

        lower_full_um = (
            coords_full * spacing
        )

        upper_full_um = (
            (
                full_shape_float
                - 1.0
                - coords_full
            )
            * spacing
        )

        distance_volume_boundary_um = float(
            np.min(
                np.concatenate(
                    [
                        lower_full_um,
                        upper_full_um,
                    ]
                )
            )
        )

        lower_roi_um = (
            coords_roi * spacing
        )

        upper_roi_um = (
            (
                roi_shape_arr
                - 1.0
                - coords_roi
            )
            * spacing
        )

        distance_patch_boundary_um = float(
            np.min(
                np.concatenate(
                    [
                        lower_roi_um,
                        upper_roi_um,
                    ]
                )
            )
        )

        predecessors = list(
            track_graph.predecessors(node_id)
        )

        successors = list(
            track_graph.successors(node_id)
        )

        backward_velocity = mean_velocity(
            node_id,
            predecessors,
            forward=False,
        )

        forward_velocity = mean_velocity(
            node_id,
            successors,
            forward=True,
        )

        detection_records.append(
            DetectionRecord(
                node_id=int(node_id),
                time_offset=(
                    local_t
                    - TARGET_LOCAL_T
                ),
                position_um=tuple(
                    float(x)
                    for x in position_rel_um
                ),
                physical_volume_um3=float(
                    physical_volume_um3
                ),
                bbox_um=tuple(
                    float(x)
                    for x in bbox_um
                ),
                pca_axes_um=tuple(
                    float(x)
                    for x in pca_axes_um
                ),
                elongation=float(feat[7]),
                flatness=float(feat[8]),
                solidity=float(feat[9]),
                compactness=float(feat[10]),
                intensity_mean=float(feat[11]),
                intensity_std=float(feat[12]),
                backward_velocity_um=tuple(
                    float(x)
                    for x in backward_velocity
                ),
                forward_velocity_um=tuple(
                    float(x)
                    for x in forward_velocity
                ),
                distance_to_volume_boundary_um=(
                    distance_volume_boundary_um
                ),
                distance_to_patch_boundary_um=(
                    distance_patch_boundary_um
                ),
                boundary_related=(
                    distance_volume_boundary_um
                    <= 4.0
                ),
            )
        )

print("Detection records:", len(detection_records))
print("Trackastra nodes :", track_graph.number_of_nodes())


In [ ]:
association_records = []

for src, dst, edge_data in track_graph.edges(data=True):
    src = int(src)
    dst = int(dst)

    relation = (
        "division"
        if track_graph.out_degree(src) > 1
        else "temporal"
    )

    score = edge_data.get(
        "weight",
        None,
    )

    if score is not None:
        score = float(score)

    association_records.append(
        AssociationRecord(
            src_node_id=src,
            dst_node_id=dst,
            score=score,
            relation=relation,
        )
    )

temporal_graph = build_temporal_graph(
    detection_records,
    association_records,
    dref_um=dref_um,
    temporal_radius=2,
    k_spatial_neighbors=6,
    spatial_radius_dref=2.5,
    current_labels=current_target,
    spacing_um=SPACING_ZYX_UM,
)

print("Associations:", len(association_records))

for key, value in temporal_graph.items():
    print(
        f"{key:25s}",
        tuple(value.shape),
        value.dtype,
    )


## Build a memory-efficient single-sample batch


In [ ]:
# Instance features for the target frame.

target_instance_meta = extract_instance_metadata(
    current_target,
    raw_norm_target,
    SPACING_ZYX_UM,
    dref_um,
    marker_target,
)

gt_ids_np = np.unique(gt_target)
gt_ids_np = gt_ids_np[gt_ids_np > 0]

spacing_np = np.asarray(
    SPACING_ZYX_UM,
    dtype=np.float32,
)

center_abs_um = (
    0.5
    * (
        np.asarray(
            gt_target.shape,
            dtype=np.float32,
        )
        - 1.0
    )
    * spacing_np
)

gt_centers_um = []

for gt_id in gt_ids_np:
    coords = np.argwhere(
        gt_target == gt_id
    )

    center_um = (
        coords.mean(axis=0)
        * spacing_np
        - center_abs_um
    )

    gt_centers_um.append(center_um)

gt_centers_um = np.asarray(
    gt_centers_um,
    dtype=np.float32,
).reshape(
    len(gt_ids_np),
    3,
)

target = {
    "ids": torch.as_tensor(
        gt_ids_np,
        dtype=torch.long,
    ),
    "centers_um": torch.as_tensor(
        gt_centers_um,
        dtype=torch.float32,
    ),
    "centers_cellscale": torch.as_tensor(
        gt_centers_um
        / max(dref_um, 1e-6),
        dtype=torch.float32,
    ),
    "label_map": torch.as_tensor(
        gt_target,
        dtype=torch.int32,
    ),
}

sample = {
    "spatial_inputs": torch.as_tensor(
        spatial_inputs_np,
        dtype=torch.float32,
    ),
    "instance_labels": torch.as_tensor(
        current_target,
        dtype=torch.int32,
    ),
    "spacing_um": torch.tensor(
        SPACING_ZYX_UM,
        dtype=torch.float32,
    ),
    "dref_um": torch.tensor(
        dref_um,
        dtype=torch.float32,
    ),
    "instance_ids": target_instance_meta.ids,
    "instance_features": target_instance_meta.features,
    "instance_centroids_um": target_instance_meta.centroids_um,
    "target": target,
    "metadata": {
        "dataset": "BlastoSPIM1",
        "series": "F22",
        "target_frame": int(TARGET_FRAME),
        "experiment": "first_overfit",
    },
}

sample.update(temporal_graph)

n_instances = len(sample["instance_ids"])
n_temporal = len(sample["temporal_ref_um"])

print("Current instances :", n_instances)
print("GT instances      :", len(target["ids"]))
print("Temporal tracklets:", n_temporal)


In [ ]:
# Single-sample batch without an extra full-volume stack/copy.

batch = {
    "spatial_inputs":
        sample["spatial_inputs"].unsqueeze(0),

    "instance_labels":
        sample["instance_labels"].unsqueeze(0),

    "spacing_um":
        sample["spacing_um"].unsqueeze(0),

    "dref_um":
        sample["dref_um"].reshape(1),

    "targets": [
        sample["target"]
    ],

    "instance_features":
        sample["instance_features"],

    "instance_ids":
        sample["instance_ids"],

    "instance_batch":
        torch.zeros(
            len(sample["instance_ids"]),
            dtype=torch.long,
        ),

    "instance_centroids_um":
        sample["instance_centroids_um"],

    "graph_x":
        sample["graph_x"],

    "graph_edge_index":
        sample["graph_edge_index"],

    "graph_edge_attr":
        sample["graph_edge_attr"],

    "tracklet_id":
        sample["tracklet_id"],

    "temporal_ref_um":
        sample["temporal_ref_um"],

    "temporal_status":
        sample["temporal_status"],

    "hypothesis_edge_index":
        sample["hypothesis_edge_index"],

    "hypothesis_edge_attr":
        sample["hypothesis_edge_attr"],

    "temporal_batch":
        torch.zeros(
            len(sample["temporal_ref_um"]),
            dtype=torch.long,
        ),
}

print("Spatial batch:", tuple(batch["spatial_inputs"].shape))
print("Graph nodes  :", tuple(batch["graph_x"].shape))
print("GT label map :", tuple(batch["targets"][0]["label_map"].shape))


## Reduced-width all-cell debug profile


In [ ]:
from learned.stirnet import StirNetConfig


cfg_debug = StirNetConfig()

cfg_debug.spatial.channels = (
    4,
    8,
    16,
    32,
)

cfg_debug.spatial.blocks_per_level = 1
cfg_debug.spatial.mask_dim = 8

cfg_debug.temporal.d_model = 32
cfg_debug.temporal.graph_heads = 4
cfg_debug.temporal.graph_ffn_dim = 64

cfg_debug.coreasoning.d_model = 32
cfg_debug.coreasoning.heads = 4
cfg_debug.coreasoning.position_bias_hidden = 8

cfg_debug.queries.d_model = 32

cfg_debug.decoder.d_model = 32
cfg_debug.decoder.heads = 4
cfg_debug.decoder.ffn_dim = 128
cfg_debug.decoder.mask_dim = 8
cfg_debug.decoder.max_spatial_tokens = 2048

required_queries = (
    2 * n_instances
    + n_temporal
    + cfg_debug.queries.discovery_queries
)

cfg_debug.queries.max_queries = max(
    512,
    int(
        math.ceil(
            required_queries / 128
        )
        * 128
    ),
)

print("Spatial channels :", cfg_debug.spatial.channels)
print("Temporal d_model :", cfg_debug.temporal.d_model)
print("Co-reasoning dim :", cfg_debug.coreasoning.d_model)
print("Decoder dim      :", cfg_debug.decoder.d_model)
print("Required queries :", required_queries)
print("Max queries      :", cfg_debug.queries.max_queries)


In [ ]:
from learned.stirnet import StirNet

from learned.stirnet.model.coreasoning import CoReasoningBlock
from learned.stirnet.model.query_builder import InstanceQueryBuilder
from learned.stirnet.model.query_decoder import InstanceQueryDecoder
from learned.stirnet.model.heads import (
    DenseAuxiliaryHeads,
    MaskEmbeddingHead,
)


model = StirNet(cfg_debug)

ch = cfg_debug.spatial.channels

model.cr1 = CoReasoningBlock(
    ch[3],
    cfg_debug.spatial,
    cfg_debug.temporal,
    cfg_debug.coreasoning,
)

model.cr2 = CoReasoningBlock(
    ch[2],
    cfg_debug.spatial,
    cfg_debug.temporal,
    cfg_debug.coreasoning,
)

model.query_builder = InstanceQueryBuilder(
    cfg_debug.queries,
    feature_channels=ch[2],
)

model.query_decoder = InstanceQueryDecoder(
    (
        ch[3],
        ch[2],
        ch[1],
    ),
    cfg_debug.decoder,
    cfg_debug.queries,
)

model.native_mask_head = MaskEmbeddingHead(
    cfg_debug.decoder.d_model,
    cfg_debug.spatial.mask_dim,
)

model.dense_heads = DenseAuxiliaryHeads(
    ch[0],
)

print("Debug STIR-Net constructed.")
print("cr1 d_model:", model.cr1.cfg.d_model)
print("cr2 d_model:", model.cr2.cfg.d_model)


## Memory-efficient criterion for the smoke overfit


In [ ]:
from torch import nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment

from learned.stirnet.model.losses import (
    binary_focal_loss_with_logits,
    dice_loss,
)
from learned.stirnet.model.matcher import (
    MatchResult,
    _pairwise_dice_cost,
    _pairwise_focal_cost,
)


class MemoryEfficientOverfitCriterion(nn.Module):
    def __init__(
        self,
        cfg,
        dense_max_voxels=500_000,
    ):
        super().__init__()

        self.cfg = cfg
        self.dense_max_voxels = int(
            dense_max_voxels
        )

        self._coarse_target_cache = {}


    def _coarse_gt(
        self,
        target,
        shape,
        device,
        dtype,
    ):
        key = tuple(
            int(x)
            for x in shape
        )

        cached = self._coarse_target_cache.get(
            key
        )

        if (
            cached is not None
            and cached.device == device
        ):
            return cached

        label_map = (
            target["label_map"]
            .float()[None, None]
        )

        labels_ds = F.interpolate(
            label_map,
            size=key,
            mode="nearest",
        )[0, 0].long()

        ids = target["ids"].long()

        masks = (
            labels_ds[None]
            == ids[:, None, None, None]
        ).to(dtype=torch.float32)

        masks = masks.to(
            device=device,
            non_blocking=True,
        )

        self._coarse_target_cache[key] = masks

        return masks


    @torch.no_grad()
    def _match(
        self,
        out,
        padding,
        targets,
    ):
        B, _ = out["exist_logits"].shape
        results = []

        for b in range(B):
            valid_q = torch.nonzero(
                ~padding[b],
                as_tuple=False,
            ).flatten()

            coarse = out[
                "coarse_mask_logits"
            ][b]

            gt_masks = self._coarse_gt(
                targets[b],
                coarse.shape[-3:],
                coarse.device,
                coarse.dtype,
            )

            gt_centers = (
                targets[b][
                    "centers_cellscale"
                ]
                .to(coarse.device)
            )

            if (
                gt_masks.shape[0] == 0
                or valid_q.numel() == 0
            ):
                results.append(
                    MatchResult(
                        valid_q[:0],
                        torch.empty(
                            0,
                            device=coarse.device,
                            dtype=torch.long,
                        ),
                    )
                )
                continue

            pl = coarse[
                valid_q
            ].flatten(1)

            gf = gt_masks.flatten(1)

            dice = _pairwise_dice_cost(
                pl,
                gf,
            )

            focal = _pairwise_focal_cost(
                pl,
                gf,
            )

            exist = (
                -F.logsigmoid(
                    out["exist_logits"][
                        b,
                        valid_q,
                    ]
                )[:, None]
            )

            center = torch.cdist(
                out["centers_cellscale"][
                    b,
                    valid_q,
                ],
                gt_centers,
                p=1,
            )

            cost = (
                2.0 * exist
                + 5.0 * dice
                + 2.0 * focal
                + 2.0 * center
            )

            row, col = linear_sum_assignment(
                cost.detach().float().cpu().numpy()
            )

            results.append(
                MatchResult(
                    valid_q[
                        torch.as_tensor(
                            row,
                            device=coarse.device,
                        )
                    ],
                    torch.as_tensor(
                        col,
                        device=coarse.device,
                        dtype=torch.long,
                    ),
                )
            )

        return results


    def _existence_loss(
        self,
        logits,
        padding,
        matches,
    ):
        target = torch.zeros_like(logits)
        valid = ~padding

        for b, match in enumerate(matches):
            target[
                b,
                match.pred_indices,
            ] = 1.0

        return binary_focal_loss_with_logits(
            logits[valid],
            target[valid],
            alpha=0.75,
            gamma=2.0,
        )


    def _coarse_losses(
        self,
        out,
        matches,
        targets,
    ):
        pred_masks = []
        gt_masks_all = []
        pred_centers = []
        gt_centers_all = []

        for b, match in enumerate(matches):
            if match.pred_indices.numel() == 0:
                continue

            pred = out[
                "coarse_mask_logits"
            ][
                b,
                match.pred_indices,
            ]

            gt = self._coarse_gt(
                targets[b],
                pred.shape[-3:],
                pred.device,
                pred.dtype,
            )[match.target_indices]

            pred_masks.append(pred)
            gt_masks_all.append(gt)

            pred_centers.append(
                out[
                    "centers_cellscale"
                ][
                    b,
                    match.pred_indices,
                ]
            )

            gt_centers_all.append(
                targets[b][
                    "centers_cellscale"
                ]
                .to(pred.device)[
                    match.target_indices
                ]
            )

        zero = (
            out["exist_logits"].sum()
            * 0.0
        )

        if not pred_masks:
            return zero, zero, zero

        pred_masks = torch.cat(
            pred_masks,
            dim=0,
        )

        gt_masks_all = torch.cat(
            gt_masks_all,
            dim=0,
        )

        pred_centers = torch.cat(
            pred_centers,
            dim=0,
        )

        gt_centers_all = torch.cat(
            gt_centers_all,
            dim=0,
        )

        return (
            dice_loss(
                pred_masks,
                gt_masks_all,
            ),
            binary_focal_loss_with_logits(
                pred_masks,
                gt_masks_all,
            ),
            F.smooth_l1_loss(
                pred_centers,
                gt_centers_all,
            ),
        )


    def _count_loss(
        self,
        logits,
        padding,
        targets,
    ):
        probability = (
            torch.sigmoid(logits)
            .masked_fill(
                padding,
                0,
            )
        )

        predicted = probability.sum(
            dim=-1
        )

        gt_count = torch.tensor(
            [
                len(target["ids"])
                for target in targets
            ],
            device=logits.device,
            dtype=logits.dtype,
        )

        return (
            F.smooth_l1_loss(
                predicted,
                gt_count,
                reduction="none",
            )
            / gt_count.clamp_min(1)
        ).mean()


    def _overlap_loss(
        self,
        coarse,
        matches,
    ):
        losses = []

        for b, match in enumerate(matches):
            if match.pred_indices.numel() < 2:
                continue

            summed = (
                coarse[
                    b,
                    match.pred_indices,
                ]
                .sigmoid()
                .sum(dim=0)
            )

            losses.append(
                F.relu(
                    summed - 1.0
                )
                .pow(2)
                .mean()
            )

        if not losses:
            return (
                coarse.sum()
                * 0.0
            )

        return torch.stack(
            losses
        ).mean()


    def _foreground_loss(
        self,
        foreground_logits,
        targets,
    ):
        _, _, Z, Y, X = (
            foreground_logits.shape
        )

        n = Z * Y * X

        if n > self.dense_max_voxels:
            scale = (
                n
                / self.dense_max_voxels
            ) ** (1.0 / 3.0)

            target_shape = tuple(
                max(
                    1,
                    int(round(v / scale)),
                )
                for v in (Z, Y, X)
            )

            pred = F.adaptive_avg_pool3d(
                foreground_logits,
                target_shape,
            )

        else:
            target_shape = (
                Z,
                Y,
                X,
            )
            pred = foreground_logits

        target_volumes = []

        for target in targets:
            label_map = (
                target["label_map"]
                .float()[None, None]
            )

            label_ds = F.interpolate(
                label_map,
                size=target_shape,
                mode="nearest",
            )[0, 0]

            target_volumes.append(
                (label_ds > 0).float()
            )

        target_fg = torch.stack(
            target_volumes,
            dim=0,
        )[:, None].to(
            pred.device
        )

        return (
            F.binary_cross_entropy_with_logits(
                pred,
                target_fg,
            )
            + dice_loss(
                pred,
                target_fg,
            )
        )


    def forward(
        self,
        outputs,
        targets,
    ):
        final = {
            "exist_logits":
                outputs.exist_logits,

            "centers_cellscale":
                outputs.centers_cellscale,

            "coarse_mask_logits":
                outputs.coarse_mask_logits,
        }

        matches = self._match(
            final,
            outputs.query_padding_mask,
            targets,
        )

        l_exist = self._existence_loss(
            outputs.exist_logits,
            outputs.query_padding_mask,
            matches,
        )

        (
            l_dice,
            l_focal,
            l_center,
        ) = self._coarse_losses(
            final,
            matches,
            targets,
        )

        l_count = self._count_loss(
            outputs.exist_logits,
            outputs.query_padding_mask,
            targets,
        )

        l_overlap = self._overlap_loss(
            outputs.coarse_mask_logits,
            matches,
        )

        l_foreground = self._foreground_loss(
            outputs.dense_outputs[
                "foreground_logits"
            ],
            targets,
        )

        total = (
            self.cfg.losses.exist
            * l_exist

            + self.cfg.losses.dice_coarse
            * l_dice

            + self.cfg.losses.focal_coarse
            * l_focal

            + self.cfg.losses.center
            * l_center

            + self.cfg.losses.count
            * l_count

            + self.cfg.losses.overlap
            * l_overlap

            + self.cfg.losses.foreground
            * l_foreground
        )

        return {
            "loss": total,
            "exist": l_exist,
            "dice_coarse": l_dice,
            "focal_coarse": l_focal,
            "center": l_center,
            "count": l_count,
            "overlap": l_overlap,
            "foreground": l_foreground,
        }


criterion = MemoryEfficientOverfitCriterion(
    cfg_debug
)

print("Memory-efficient overfit criterion ready.")


## Move the single sample to the GPU


In [ ]:
# Stabilize highly skewed shape-ratio graph features.
#
# graph_x columns:
#   11 = elongation
#   12 = flatness
#
# Ratios can become extremely large for nearly degenerate
# components when one PCA axis approaches zero.

graph_x_fixed = batch["graph_x"].clone()

for col in (11, 12):
    graph_x_fixed[:, col] = torch.log1p(
        graph_x_fixed[:, col].clamp_min(0.0)
    )

batch["graph_x"] = graph_x_fixed


print("After shape-ratio stabilization:")

print(
    "elongation max:",
    float(batch["graph_x"][:, 11].max()),
)

print(
    "flatness max  :",
    float(batch["graph_x"][:, 12].max()),
)

In [ ]:
from learned.stirnet.training.trainer import (
    move_to_device,
    model_forward_from_batch,
)


# Clean up any stale CUDA allocations from prior experimentation.
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

device = torch.device("cuda")

batch_device = {}

for key, value in batch.items():
    if key == "targets":
        # Keep the label-map targets on CPU.
        batch_device[key] = value

    elif key == "spatial_inputs":
        batch_device[key] = value.to(
            device=device,
            dtype=torch.float16,
        )

    elif key == "instance_labels":
        batch_device[key] = value.to(
            device=device,
            dtype=torch.int32,
        )

    else:
        batch_device[key] = move_to_device(
            value,
            device,
        )

model = model.to(device)
criterion = criterion.to(device)

print(
    "CUDA allocated before forward:",
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB",
)


In [ ]:
gx = batch_device["graph_x"].detach().float()

print(
    "GPU elongation max:",
    float(gx[:, 11].max()),
)

print(
    "GPU flatness max:",
    float(gx[:, 12].max()),
)

print(
    "All graph_x finite:",
    bool(torch.isfinite(gx).all()),
)

In [ ]:
model.eval()

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):

        node_embeddings = model.graph_encoder(
            batch_device["graph_x"],
            batch_device["graph_edge_index"],
            batch_device["graph_edge_attr"],
        )


x = node_embeddings.detach().float()

print("Graph encoder output")
print("--------------------")
print("shape :", tuple(x.shape))
print("finite:", bool(torch.isfinite(x).all()))
print("nan   :", int(torch.isnan(x).sum()))
print("inf   :", int(torch.isinf(x).sum()))

if torch.isfinite(x).all():
    print(
        "range :",
        float(x.min()),
        "to",
        float(x.max()),
    )

In [ ]:
import math

import torch
import torch.nn.functional as F

from learned.stirnet.model.query_decoder import (
    QueryCrossAttention,
)


def amp_safe_query_cross_attention_forward(
    self,
    q_in,
    spatial,
    spatial_pos_um,
    refs_um,
    support,
    query_padding_mask,
    dref_um,
):
    B, Q, _ = q_in.shape

    # Keep the output in the same dtype as the query state.
    out = torch.zeros_like(q_in)

    for b in range(B):

        valid_q = ~query_padding_mask[b]

        if not valid_q.any():
            continue

        q = self._split(
            self.q(
                q_in[b, valid_q]
            )
        ).permute(1, 0, 2)

        k = self._split(
            self.k(
                spatial[b]
            )
        ).permute(1, 0, 2)

        v = self._split(
            self.v(
                spatial[b]
            )
        ).permute(1, 0, 2)

        logits = torch.einsum(
            "hqd,hnd->hqn",
            q,
            k,
        ) / math.sqrt(
            self.head_dim
        )

        delta = (
            spatial_pos_um[b][None]
            - refs_um[
                b,
                valid_q,
            ][:, None]
        )

        logits = (
            logits
            + self.pos_bias(
                delta,
                dref_um[b],
            ).permute(2, 0, 1)
        )

        sup = support[
            b,
            valid_q,
        ].clone()

        # Guarantee at least one valid spatial token
        # for every query.
        empty = ~sup.any(
            dim=-1
        )

        if empty.any():

            nearest = (
                torch.linalg.vector_norm(
                    delta[empty],
                    dim=-1,
                )
                .argmin(dim=-1)
            )

            rows = torch.nonzero(
                empty,
                as_tuple=False,
            ).flatten()

            sup[
                rows,
                nearest,
            ] = True

        logits = logits.masked_fill(
            ~sup[None],
            -1e4,
        )

        weights = torch.softmax(
            logits,
            dim=-1,
        )

        weights = F.dropout(
            weights,
            self.dropout,
            self.training,
        )

        msg = torch.einsum(
            "hqn,hnd->hqd",
            weights,
            v,
        )

        msg = (
            msg
            .permute(1, 0, 2)
            .reshape(
                int(valid_q.sum()),
                self.d_model,
            )
        )

        projected = self.out(msg)

        # Critical AMP fix:
        # destination is q_in.dtype, while autocast may
        # produce projected as float16.
        projected = projected.to(
            dtype=out.dtype
        )

        out[
            b,
            valid_q,
        ] = projected

    return out


QueryCrossAttention.forward = (
    amp_safe_query_cross_attention_forward
)

print(
    "AMP-safe QueryCrossAttention installed."
)

## Forward gate


In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.eval()

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        outputs = model_forward_from_batch(
            model,
            batch_device,
        )

        initial_losses = criterion(
            outputs,
            batch_device["targets"],
        )

initial_metrics = {
    key: float(
        value.detach().cpu()
    )
    for key, value in initial_losses.items()
}

print("Initial loss")
print("------------")

for key, value in initial_metrics.items():
    print(
        f"{key:16s}: {value:.6f}"
    )

print()
print(
    "Peak GPU memory:",
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB",
)


In [ ]:
def finite_report(name, x):
    if not torch.is_tensor(x):
        return

    y = x.detach().float()

    finite = torch.isfinite(y)

    print(
        f"{name:28s} "
        f"shape={str(tuple(x.shape)):20s} "
        f"dtype={str(x.dtype):12s} "
        f"finite={bool(finite.all())} "
        f"nan={int(torch.isnan(y).sum())} "
        f"inf={int(torch.isinf(y).sum())}"
    )

    if finite.any():
        vals = y[finite]

        print(
            f"{'':28s} "
            f"min={float(vals.min()):.5g} "
            f"max={float(vals.max()):.5g}"
        )


keys_to_check = [
    "spatial_inputs",
    "spacing_um",
    "dref_um",

    "instance_features",
    "instance_centroids_um",

    "graph_x",
    "graph_edge_attr",

    "temporal_ref_um",
    "temporal_status",

    "hypothesis_edge_attr",
]


for key in keys_to_check:
    finite_report(
        key,
        batch_device[key],
    )

In [ ]:
import torch


def iter_tensors(obj):
    if torch.is_tensor(obj):
        yield obj

    elif isinstance(obj, (list, tuple)):
        for item in obj:
            yield from iter_tensors(item)

    elif isinstance(obj, dict):
        for item in obj.values():
            yield from iter_tensors(item)

    elif hasattr(obj, "__dict__"):
        for item in vars(obj).values():
            yield from iter_tensors(item)


handles = []


def make_nonfinite_hook(name):

    def hook(module, inputs, output):

        tensors = list(
            iter_tensors(output)
        )

        for i, tensor in enumerate(tensors):

            if tensor.numel() == 0:
                continue

            x = tensor.detach().float()

            if not torch.isfinite(x).all():

                nan_count = int(
                    torch.isnan(x).sum()
                )

                inf_count = int(
                    torch.isinf(x).sum()
                )

                raise RuntimeError(
                    "\nFIRST NON-FINITE MODULE\n"
                    f"name   : {name}\n"
                    f"type   : {module.__class__.__name__}\n"
                    f"tensor : {i}\n"
                    f"shape  : {tuple(tensor.shape)}\n"
                    f"dtype  : {tensor.dtype}\n"
                    f"nan    : {nan_count}\n"
                    f"inf    : {inf_count}"
                )

    return hook


for name, module in model.named_modules():

    # Hook leaf modules so the error points as close as
    # possible to the actual operation.
    if len(list(module.children())) == 0:

        handles.append(
            module.register_forward_hook(
                make_nonfinite_hook(name)
            )
        )


print(
    "Installed",
    len(handles),
    "non-finite diagnostic hooks."
)

In [ ]:
model.eval()

try:

    with torch.no_grad():

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):

            debug_outputs = model_forward_from_batch(
                model,
                batch_device,
            )

    print("No non-finite leaf-module output detected.")

finally:

    for handle in handles:
        handle.remove()

In [ ]:
graph_feature_names = [
    "time_offset",                 # 0
    "pos_z",                       # 1
    "pos_y",                       # 2
    "pos_x",                       # 3
    "log_volume_ratio",            # 4
    "bbox_z",                      # 5
    "bbox_y",                      # 6
    "bbox_x",                      # 7
    "pca_axis_1",                  # 8
    "pca_axis_2",                  # 9
    "pca_axis_3",                  # 10
    "elongation",                  # 11
    "flatness",                    # 12
    "solidity",                    # 13
    "compactness",                 # 14
    "intensity_mean",              # 15
    "intensity_std",               # 16
    "back_vel_z",                  # 17
    "back_vel_y",                  # 18
    "back_vel_x",                  # 19
    "forward_vel_z",               # 20
    "forward_vel_y",               # 21
    "forward_vel_x",               # 22
    "track_length_before",         # 23
    "track_length_after",          # 24
    "distance_volume_boundary",    # 25
    "distance_patch_boundary",     # 26
    "is_current",                  # 27
    "interior_start",              # 28
    "interior_end",                # 29
    "division",                    # 30
    "boundary_related",            # 31
]


gx = batch["graph_x"].detach().float().cpu()

print(
    f"{'idx':>3s} "
    f"{'feature':28s} "
    f"{'min':>14s} "
    f"{'max':>14s} "
    f"{'max_abs':>14s}"
)

print("-" * 78)

for i, name in enumerate(graph_feature_names):

    col = gx[:, i]

    print(
        f"{i:3d} "
        f"{name:28s} "
        f"{float(col.min()):14.4g} "
        f"{float(col.max()):14.4g} "
        f"{float(col.abs().max()):14.4g}"
    )

In [ ]:
print("FP16 maximum finite value:", torch.finfo(torch.float16).max)

for i, name in enumerate(graph_feature_names):

    max_abs = float(
        gx[:, i].abs().max()
    )

    if max_abs > 1000:
        print(
            f"VERY LARGE: "
            f"{i:2d} {name:28s} "
            f"{max_abs:.6g}"
        )

## Same-sample overfit

This is intentionally not a generalization test. The exact same sample is optimized
repeatedly. The first success criterion is simply that the evaluation cost falls clearly.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)

OVERFIT_STEPS = 100
EVAL_EVERY = 5


def evaluate():
    model.eval()

    with torch.no_grad():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            out = model_forward_from_batch(
                model,
                batch_device,
            )

            losses = criterion(
                out,
                batch_device["targets"],
            )

    return {
        key: float(
            value.detach().cpu()
        )
        for key, value in losses.items()
    }


history = [
    {
        "step": 0,
        **initial_metrics,
    }
]

start_time = time.perf_counter()

for step in range(
    1,
    OVERFIT_STEPS + 1,
):
    model.train()

    optimizer.zero_grad(
        set_to_none=True,
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        out = model_forward_from_batch(
            model,
            batch_device,
        )

        losses = criterion(
            out,
            batch_device["targets"],
        )

        loss = losses["loss"]

    scaler.scale(
        loss
    ).backward()

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0,
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    if (
        step == 1
        or step % EVAL_EVERY == 0
        or step == OVERFIT_STEPS
    ):
        metrics = evaluate()

        history.append(
            {
                "step": step,
                **metrics,
            }
        )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        print(
            f"step={step:3d} | "
            f"loss={metrics['loss']:.5f} | "
            f"time={elapsed:.1f}s"
        )


## Plot and summarize the cost reduction


In [ ]:
import matplotlib.pyplot as plt


steps = np.asarray(
    [
        row["step"]
        for row in history
    ]
)

losses = np.asarray(
    [
        row["loss"]
        for row in history
    ]
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    steps,
    losses,
    marker="o",
)

plt.xlabel(
    "Optimization step"
)

plt.ylabel(
    "Evaluation loss"
)

plt.title(
    "STIR-Net first single-sample overfit"
)

plt.grid(
    alpha=0.25
)

plt.show()

initial_loss = float(
    losses[0]
)

final_loss = float(
    losses[-1]
)

reduction_pct = (
    100.0
    * (
        initial_loss
        - final_loss
    )
    / max(
        initial_loss,
        1e-12,
    )
)

print("Initial loss:", f"{initial_loss:.6f}")
print("Final loss  :", f"{final_loss:.6f}")
print("Reduction   :", f"{reduction_pct:.2f}%")

print()
print(
    "PASS"
    if final_loss < initial_loss
    else "FAIL"
)


In [ ]:
print(
    f"{'component':18s}"
    f"{'initial':>14s}"
    f"{'final':>14s}"
    f"{'change':>14s}"
)

print("-" * 60)

for name in [
    "exist",
    "dice_coarse",
    "focal_coarse",
    "center",
    "count",
    "overlap",
    "foreground",
]:
    initial = float(
        history[0][name]
    )

    final = float(
        history[-1][name]
    )

    print(
        f"{name:18s}"
        f"{initial:14.6f}"
        f"{final:14.6f}"
        f"{final - initial:14.6f}"
    )
